# RegionAnalyzer — synthetic 3D segmentation

Build a small labeled volume (three non-overlapping objects) and extract region properties as a pandas DataFrame.

In [1]:
import numpy as np
import pandas as pd
import stackview

from vistiq.constant.matrix import FULL, LOWER, LOWER_ND, OFF_DIAGONAL, UPPER, UPPER_ND
from vistiq.utils import ArrayIteratorConfig
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig, region_to_numpy, dataframe_to_numpy
from vistiq.segment.select import RegionFilter, RegionFilterConfig, RangeFilterConfig, ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig

2026-07-06 15:35:01,856 - INFO - No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


## Synthetic label volume

Shape `(100, 100, 10)` with labels `1`, `2`, and `3` in separate spatial regions (no overlap), mimicking a 3D segmentation mask.

In [2]:
labels = np.zeros((10, 200, 200), dtype=np.uint64)

# Object 1 — upper-left
labels[2:7, 20:38, 12:30] = 1

# Object 2 — center
labels[3:9, 42:58, 38:62] = 2

# Object 3 — lower-right
labels[1:6, 72:92, 68:88] = 3

# Object 4 — lower-right
labels[4:6, 112:132, 125:163] = 4

# Object 5 — lower-right
labels[7:9, 145:180, 12:58] = 5

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

labels.shape=(10, 200, 200), dtype=uint64
unique labels: [0 1 2 3 4 5]
voxel counts: {1: 1620, 2: 2304, 3: 2000, 4: 1520, 5: 3220}


In [3]:
areas = np.zeros((10, 200, 200), dtype=np.uint64)

# Area 1 — upper-left
areas[2:9, 10:58, 10:98] = 6


# Area2 — lower-right
areas[1:10, 96:192, 58:178] = 8

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

labels.shape=(10, 200, 200), dtype=uint64
unique labels: [0 1 2 3 4 5]
voxel counts: {1: 1620, 2: 2304, 3: 2000, 4: 1520, 5: 3220}


In [4]:
stackview.slice(np.concatenate([labels, areas], axis=-1))

## RegionAnalyzer (dataframe output)

Analyze the full 3D volume (`slice_def=()`). With `map_axes=True`, vector properties such as `cross_sectional_area` and `aspect_ratio` are expanded to plane-specific columns (`-xy`, `-xz`, `-yz`).

In [5]:
metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
}

config = RegionAnalyzerConfig(
    output_type="dataframe",
    map_axes=True,
    properties=[
        "label",
        "volume",
        "centroid",
        "bbox",
        "aspect_ratio",
        "cross_sectional_area",
    ],
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

l_regions = RegionAnalyzer(config).run(labels, metadata=metadata)
a_regions = RegionAnalyzer(config).run(areas, metadata=metadata)
all_regions = pd.concat([l_regions, a_regions], axis=0)

#print(f"{len(l_regions)} regions, {len(l_regions.columns)} columns")
#print(f"{len(a_regions)} areas, {len(a_regions.columns)} columns")

2026-07-06 15:35:03,139 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,185 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,198 - INFO - Running RegionAnalyzer with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None index_on='label' properties=['label', 'stack_id', 'slice_id', 'object_id', 'centroid', 'bbox', 'aspect_ratio', 'cross_sectional_area', 'area'] map_ax

In [6]:
all_regions

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
1,8.0,28.5,20.5,2,20,12,7,38,30,3240.0,0.545173,0.545173,1.000000,0.545173,180.0,180.0,324.0,c493a77854824e4a8f118c1d3cb6ea88,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
2,11.0,49.5,49.5,3,42,38,9,58,62,4608.0,0.740959,0.493435,0.665942,0.493435,192.0,288.0,384.0,a76805674d454b84a577250d5528539e,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
3,6.0,81.5,77.5,1,72,68,6,92,88,4000.0,0.490511,0.490511,1.000000,0.490511,200.0,200.0,400.0,71d67bdbd368499e875bf970dbfe1f15,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
4,9.0,121.5,143.5,4,112,125,6,132,163,3040.0,0.173422,0.091192,0.525840,0.091192,80.0,152.0,760.0,9bc23a476a014019a315f37cb2286c02,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
5,15.0,162.0,34.5,7,145,12,9,180,58,6440.0,0.099015,0.075324,0.760739,0.075324,140.0,184.0,1610.0,f011c8a5adcd4882be49b045f303a7a4,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
6,10.0,33.5,53.5,2,10,10,9,58,98,59136.0,0.288738,0.157469,0.545371,0.157469,672.0,1232.0,4224.0,761dfd19337a4671be7d3520ad4b30d7,04277d0900bc40049b89e4f4b1e0443c,88740e0891834496b83f6247ef04c76d
8,10.0,143.5,117.5,1,96,58,10,192,178,207360.0,0.186349,0.149076,0.799984,0.149076,1728.0,2160.0,11520.0,77ee818796ba4f449cf15c846d19d889,04277d0900bc40049b89e4f4b1e0443c,88740e0891834496b83f6247ef04c76d


In [7]:
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="volume",
            range=(100.0,np.inf)
        ),
        #MinFilterConfig(
        #    attribute="aspect_ratio",
        #    minimum=0.015,
        #),
    ]
)
l_accepted, _ = RegionFilter(rfcfg).run(l_regions)
a_accepted, _ = RegionFilter(rfcfg).run(a_regions)
all_accepted = pd.concat([l_accepted, a_accepted], axis=0)

2026-07-06 15:35:03,422 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,467 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,469 - INFO - Running RegionFilter with config: classname='Configurable' package='vistiq.core' version=None command_group=None filters=[RangeFilterConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, attribute='volume', axis=None, strict=True, preferred_input_type='numpy', range=(100.0, inf))]
2026-07-06 15:35:03,470 - INFO - Applying RegionFilter to a DataFrame
2026-07-06 15:35:03,471 - INFO - RegionFilter: len(accepted_regions)=5, len(removed_labels)=0
2026-07-06 15:35:03,472 - INFO - Finished in state Comple

In [8]:
all_accepted

,centroid-z,centroid-y,centroid-x,bbox-start-z,bbox-start-y,bbox-start-x,bbox-end-z,bbox-end-y,bbox-end-x,volume,aspect_ratio-yz,aspect_ratio-xz,aspect_ratio-xy,aspect_ratio,cross_sectional_area-yz,cross_sectional_area-xz,cross_sectional_area-xy,object_id,stack_id,slice_id
label,,,,,,,,,,,,,,,,,,,,
1,8.0,28.5,20.5,2,20,12,7,38,30,3240.0,0.545173,0.545173,1.000000,0.545173,180.0,180.0,324.0,c493a77854824e4a8f118c1d3cb6ea88,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
2,11.0,49.5,49.5,3,42,38,9,58,62,4608.0,0.740959,0.493435,0.665942,0.493435,192.0,288.0,384.0,a76805674d454b84a577250d5528539e,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
3,6.0,81.5,77.5,1,72,68,6,92,88,4000.0,0.490511,0.490511,1.000000,0.490511,200.0,200.0,400.0,71d67bdbd368499e875bf970dbfe1f15,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
4,9.0,121.5,143.5,4,112,125,6,132,163,3040.0,0.173422,0.091192,0.525840,0.091192,80.0,152.0,760.0,9bc23a476a014019a315f37cb2286c02,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
5,15.0,162.0,34.5,7,145,12,9,180,58,6440.0,0.099015,0.075324,0.760739,0.075324,140.0,184.0,1610.0,f011c8a5adcd4882be49b045f303a7a4,2428306416dd4346ac316202076017ad,c9c71989b3594c29a6c2367c7d817fdb
6,10.0,33.5,53.5,2,10,10,9,58,98,59136.0,0.288738,0.157469,0.545371,0.157469,672.0,1232.0,4224.0,761dfd19337a4671be7d3520ad4b30d7,04277d0900bc40049b89e4f4b1e0443c,88740e0891834496b83f6247ef04c76d
8,10.0,143.5,117.5,1,96,58,10,192,178,207360.0,0.186349,0.149076,0.799984,0.149076,1728.0,2160.0,11520.0,77ee818796ba4f449cf15c846d19d889,04277d0900bc40049b89e4f4b1e0443c,88740e0891834496b83f6247ef04c76d


## Calculate distances between all centroids

In [9]:
from vistiq.analysis import DistanceCalculator, DistanceCalculatorConfig

dccfg = DistanceCalculatorConfig(
    annotate=True, 
    output_type="dataframe" #"torch.Tensor"
)

centroids = dataframe_to_numpy(all_accepted, attributes=["centroid"], strict=False)
object_ids = dataframe_to_numpy(all_accepted, attributes=["object_id"])
dist = DistanceCalculator(dccfg).run(centroids, centroids, spacing=metadata.get("scale", None), point_annotations=(object_ids, object_ids))

2026-07-06 15:35:03,667 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,703 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,724 - INFO - Found mps device: Apple Metal (MPS)
2026-07-06 15:35:03,790 - INFO - MatrixCalculator.run: device=mps, points1.shape=torch.Size([7, 3]), points1.dtype=torch.float32, points2.shape=torch.Size([7, 3]), points2.dtype=torch.float32, spacing=(2.0, 1.0, 1.0)
2026-07-06 15:35:03,799 - INFO - DistanceCalculator._calculate: distances.shape=torch.Size([7, 7])
2026-07-06 15:35:03,801 - INFO - Finished in state Completed()


In [10]:
dist

,c493a77854824e4a8f118c1d3cb6ea88,a76805674d454b84a577250d5528539e,71d67bdbd368499e875bf970dbfe1f15,9bc23a476a014019a315f37cb2286c02,f011c8a5adcd4882be49b045f303a7a4,761dfd19337a4671be7d3520ad4b30d7,77ee818796ba4f449cf15c846d19d889
c493a77854824e4a8f118c1d3cb6ea88,0.000000,36.304268,77.935867,154.214142,134.960175,33.615471,150.499176
a76805674d454b84a577250d5528539e,36.304268,0.000000,43.680660,118.473625,113.777191,16.613247,116.034477
71d67bdbd368499e875bf970dbfe1f15,77.935867,43.680660,0.000000,77.408012,93.022850,54.258640,74.215904
9bc23a476a014019a315f37cb2286c02,154.214142,118.473625,77.408012,0.000000,116.898460,125.888840,34.117443
f011c8a5adcd4882be49b045f303a7a4,134.960175,113.777191,93.022850,116.898460,0.000000,130.281433,85.622719
761dfd19337a4671be7d3520ad4b30d7,33.615471,16.613247,54.258640,125.888840,130.281433,0.000000,127.263504
77ee818796ba4f449cf15c846d19d889,150.499176,116.034477,74.215904,34.117443,85.622719,127.263504,0.000000


## Filter to get k smallest/largest value along axis

If the filter is applied to a distance matrix, it can be used to get the k-nearest neighbors. **Important:** set `triangle=OFF_DIAGONAL`.

In [11]:
from vistiq.constant.matrix import DIAGONAL, OFF_DIAGONAL, UPPER

tkcfg = TopKFilterConfig(
    k=1,
    axis=1,
    largest=False,
    triangle=OFF_DIAGONAL,
    output="masked_values",
)

tk = TopKFilter(tkcfg).run(dist.to_numpy())
tk

2026-07-06 15:35:03,891 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,932 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:03,934 - INFO - Found mps device: Apple Metal (MPS)
2026-07-06 15:35:04,001 - INFO - Finished in state Completed()


tensor([[    nan,     nan,     nan,     nan,     nan, 33.6155,     nan],
        [    nan,     nan,     nan,     nan,     nan, 16.6132,     nan],
        [    nan, 43.6807,     nan,     nan,     nan,     nan,     nan],
        [    nan,     nan,     nan,     nan,     nan,     nan, 34.1174],
        [    nan,     nan,     nan,     nan,     nan,     nan, 85.6227],
        [    nan, 16.6132,     nan,     nan,     nan,     nan,     nan],
        [    nan,     nan,     nan, 34.1174,     nan,     nan,     nan]],
       device='mps:0')

In [12]:
import torch

mincfg = ValueFilterConfig(
    ref_value=80.0,
    axis=0,
    operator=">",
    triangle=LOWER_ND,
    output="masked_values",
)
maxcfg = ValueFilterConfig(
    ref_value=120.0,
    axis=0,
    operator="<",
    triangle=LOWER_ND,
    output="masked_values",
)
mint = ValueFilter(mincfg).run(dist.to_numpy())
maxt = ValueFilter(maxcfg).run(dist.to_numpy())
ranget = torch.eq(mint,maxt)
mint, maxt, ranget

2026-07-06 15:35:04,148 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,201 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,203 - INFO - Found mps device: Apple Metal (MPS)
2026-07-06 15:35:04,228 - INFO - Finished in state Completed()
2026-07-06 15:35:04,295 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,344 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 1

(tensor([[     nan,      nan,      nan,      nan,      nan,      nan,      nan],
         [     nan,      nan,      nan,      nan,      nan,      nan,      nan],
         [     nan,      nan,      nan,      nan,      nan,      nan,      nan],
         [154.2141, 118.4736,      nan,      nan,      nan,      nan,      nan],
         [134.9602, 113.7772,  93.0229, 116.8985,      nan,      nan,      nan],
         [     nan,      nan,      nan, 125.8888, 130.2814,      nan,      nan],
         [150.4992, 116.0345,      nan,      nan,  85.6227, 127.2635,      nan]],
        device='mps:0'),
 tensor([[     nan,      nan,      nan,      nan,      nan,      nan,      nan],
         [ 36.3043,      nan,      nan,      nan,      nan,      nan,      nan],
         [ 77.9359,  43.6807,      nan,      nan,      nan,      nan,      nan],
         [     nan, 118.4736,  77.4080,      nan,      nan,      nan,      nan],
         [     nan, 113.7772,  93.0229, 116.8985,      nan,      nan,      nan],
  

In [13]:
from vistiq.analysis.matrix import MatrixAggregatorConfig, MatrixAggregator

macfg = MatrixAggregatorConfig(
    operation="sum",
    axis=1,
)

counts = MatrixAggregator(macfg).run(ranget>0)
counts

2026-07-06 15:35:04,457 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,521 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,534 - INFO - Finished in state Completed()


tensor([0, 0, 0, 1, 3, 0, 2], device='mps:0')

## Build containment graph

In [14]:
from vistiq.analysis import OverlapCalculator, LabelOverlapCalculatorConfig, IoUMetricsCalculatorConfig, IoSMetricsCalculatorConfig, DiceMetricsCalculatorConfig, region_map_from_dataframe

a_object_ids = dataframe_to_numpy(a_accepted, attributes=["object_id"])
l_object_ids = dataframe_to_numpy(l_accepted, attributes=["object_id"])

# Calculate overlap metrics b
loc = LabelOverlapCalculatorConfig(
    metrics_calculators=[
        #IoUMetricsCalculatorConfig(), 
        IoSMetricsCalculatorConfig(),
        #DiceMetricsCalculatorConfig(),
    ],
    return_components=False,
    annotate=True,
    output_type="dataframe",
)
oc =  OverlapCalculator(loc)
overlap = oc.run(
    areas, 
    labels, 
    region_map=(
        region_map_from_dataframe(a_accepted.reset_index()),
        region_map_from_dataframe(l_accepted.reset_index()),
    ),
)

# format into dataframe
ios = oc.format(overlap, metric="ios")
ios

2026-07-06 15:35:04,609 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,648 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,658 - INFO - Running OverlapCalculator with config: classname='Configurable' package='vistiq.core' version=None command_group=None builder=LabelBuilderConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, preferred_input_type='numpy', preferred_device=None) area_calculator=LabelAreaCalculatorConfig(classname='Configurable', package='vistiq.core', version=None, command_group=None, preferred_input_type='numpy', preferred_device=None) intersection_calculator=LabelIntersectionCalculatorConfig(classname='Configurabl

,c493a77854824e4a8f118c1d3cb6ea88,a76805674d454b84a577250d5528539e,71d67bdbd368499e875bf970dbfe1f15,9bc23a476a014019a315f37cb2286c02,f011c8a5adcd4882be49b045f303a7a4
761dfd19337a4671be7d3520ad4b30d7,1.0,1.0,0.0,0.0,0.0
77ee818796ba4f449cf15c846d19d889,0.0,0.0,0.0,1.0,0.0


## Filter IoS matrix

A label is considered to be contained within another label if the IoS>0.5.

In [15]:
vcfg = ValueFilterConfig(
    ref_value=0.5,
    operator=">",
    output="masked_values",
)
ios_filtered = ValueFilter(vcfg).run(overlap.metrics["ios"])
ios_filtered

2026-07-06 15:35:04,881 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,922 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:04,923 - INFO - Found mps device: Apple Metal (MPS)
2026-07-06 15:35:04,954 - INFO - Finished in state Completed()


tensor([[1., 1., nan, nan, nan],
        [nan, nan, nan, 1., nan]], device='mps:0')

In [16]:
from vistiq.graph import GraphBuilder, GraphBuilderConfig

gbc = GraphBuilderConfig(
    edge_attribute = "ios",
)
df = pd.DataFrame(np.array(ios_filtered.cpu()), columns=l_accepted["object_id"], index=a_accepted["object_id"])
G = GraphBuilder(gbc).run(df, all_regions)
G

2026-07-06 15:35:05,054 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:05,092 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:35:05,096 - INFO - Building graph with config: classname='Configurable' package='vistiq.core' version=None command_group=None edge_attribute='ios' synthetic_attribute='synthetic' graph_type='directed'
2026-07-06 15:35:05,096 - INFO - Matrix shape: (2, 5)
2026-07-06 15:35:05,096 - INFO - Regions shape: (7, 20)
2026-07-06 15:35:05,099 - INFO - Finished in state Completed()


## GraphLike

`GraphBuilder.run` returns a `GraphLike` graph. The interface mirrors networkx, but node and edge
attributes are read through `node_attrs` / `edge_attrs` instead of `graph.nodes[id]`.

In [23]:
from vistiq.graph import GraphLike, NXGraph

assert isinstance(G, GraphLike)

print(
    f"nodes={G.number_of_nodes()}, "
    f"edges={G.number_of_edges()}, "
    f"directed={G.is_directed()}, "
    f"longest path={G.dag_longest_path()}, ",
    f"all_pairs_shortest_path_length={G.all_pairs_shortest_path_length()}.", 
    #f"single_source_shortest_path_length={G.single_source_shortest_path_length()}"
)

sample_id = next(iter(G.nodes()))
print("sample node:", sample_id)
G.node_attrs(sample_id)

source, target = next(iter(G.edges()))
print(f"sample edge {source!r} -> {target!r}:", G.edge_attrs(source, target))

print(f"density={G.density():.3f}")

# unwrap the networkx object when needed
type(G.raw).__name__ if isinstance(G, NXGraph) else None

nodes=7, edges=3, directed=True, longest path=['761dfd19337a4671be7d3520ad4b30d7', 'c493a77854824e4a8f118c1d3cb6ea88'],  all_pairs_shortest_path_length={'761dfd19337a4671be7d3520ad4b30d7': {'761dfd19337a4671be7d3520ad4b30d7': 0, 'c493a77854824e4a8f118c1d3cb6ea88': 1, 'a76805674d454b84a577250d5528539e': 1}, '77ee818796ba4f449cf15c846d19d889': {'77ee818796ba4f449cf15c846d19d889': 0, '9bc23a476a014019a315f37cb2286c02': 1}, 'c493a77854824e4a8f118c1d3cb6ea88': {'c493a77854824e4a8f118c1d3cb6ea88': 0}, 'a76805674d454b84a577250d5528539e': {'a76805674d454b84a577250d5528539e': 0}, '71d67bdbd368499e875bf970dbfe1f15': {'71d67bdbd368499e875bf970dbfe1f15': 0}, '9bc23a476a014019a315f37cb2286c02': {'9bc23a476a014019a315f37cb2286c02': 0}, 'f011c8a5adcd4882be49b045f303a7a4': {'f011c8a5adcd4882be49b045f303a7a4': 0}}.
sample node: 761dfd19337a4671be7d3520ad4b30d7
sample edge '761dfd19337a4671be7d3520ad4b30d7' -> 'c493a77854824e4a8f118c1d3cb6ea88': {'ios': 1.0}
density=0.071


'DiGraph'

## GraphQuery

Request summary keys via `GraphQueryConfig.attributes`. Defaults cover counts, roots/leaves,
parent/child maps, and edge records.

In [18]:
from vistiq.graph import GraphQuery, GraphQueryConfig

gq = GraphQuery(
    GraphQueryConfig(
        attributes=[
            "n_nodes",
            "n_edges",
            "roots",
            "leaves",
            "nodes_by_attribute",
            "edges",
        ],
        group_attribute="label",
        weight_attribute="ios",
    )
)
summary = gq.run(G)
{k: v for k, v in summary.items()}


2026-07-06 15:36:00,651 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:36:00,694 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:36:00,699 - INFO - Summarizing graph with config: classname='Configurable' package='vistiq.core' version=None command_group=None label_attribute='object_name' group_attribute='label' filter_attribute=None filter_value=None include_attributes=[] lineage_value_attribute='label' weight_attribute='ios' neighbor_analysis=None neighbor_k=None neighbor_radius=None source_filter=None target_filter=None source_nodes=None predecessor_match=None seed_nodes=None attributes=['n_nodes', 'n_edges', 'roots', 'leaves', 'nodes_by_attribute', 'edges'] output_type='list' 

{'n_nodes': 7,
 'n_edges': 3,
 'roots': ['71d67bdbd368499e875bf970dbfe1f15',
  '761dfd19337a4671be7d3520ad4b30d7',
  '77ee818796ba4f449cf15c846d19d889',
  'f011c8a5adcd4882be49b045f303a7a4'],
 'leaves': ['71d67bdbd368499e875bf970dbfe1f15',
  '9bc23a476a014019a315f37cb2286c02',
  'a76805674d454b84a577250d5528539e',
  'c493a77854824e4a8f118c1d3cb6ea88',
  'f011c8a5adcd4882be49b045f303a7a4'],
 'nodes_by_attribute': {},
 'edges': [{'parent': '761dfd19337a4671be7d3520ad4b30d7',
   'child': 'c493a77854824e4a8f118c1d3cb6ea88',
   'ios': 1.0},
  {'parent': '761dfd19337a4671be7d3520ad4b30d7',
   'child': 'a76805674d454b84a577250d5528539e',
   'ios': 1.0},
  {'parent': '77ee818796ba4f449cf15c846d19d889',
   'child': '9bc23a476a014019a315f37cb2286c02',
   'ios': 1.0}]}

In [19]:
pd.DataFrame(summary["edges"])#.head()


,parent,child,ios
0,761dfd19337a4671be7d3520ad4b30d7,c493a77854824e4a8f118c1d3cb6ea88,1.0
1,761dfd19337a4671be7d3520ad4b30d7,a76805674d454b84a577250d5528539e,1.0
2,77ee818796ba4f449cf15c846d19d889,9bc23a476a014019a315f37cb2286c02,1.0


In [20]:
seed_id = l_regions["object_id"].iloc[0]
partner_edges = GraphQuery(
    GraphQueryConfig(
        attributes=["filtered_edges"],
        source_nodes=[seed_id],
    )
).run(G)["filtered_edges"]
pd.DataFrame(partner_edges).head()


2026-07-06 15:36:04,906 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:36:04,953 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:36:04,959 - INFO - Summarizing graph with config: classname='Configurable' package='vistiq.core' version=None command_group=None label_attribute='object_name' group_attribute='channel' filter_attribute=None filter_value=None include_attributes=[] lineage_value_attribute='label' weight_attribute='ios' neighbor_analysis=None neighbor_k=None neighbor_radius=None source_filter=None target_filter=None source_nodes=['c493a77854824e4a8f118c1d3cb6ea88'] predecessor_match=None seed_nodes=None attributes=['filtered_edges'] output_type='list' output_index='object

""


## GraphFilter

Select node ids or edge records by id list or attribute match. Modes: `nodes`, `edges`,
`direct_path`, `full_path`.

In [48]:
from vistiq.graph import GraphFilter, GraphFilterConfig

largest_id = l_regions.loc[l_regions["volume"].idxmax(), "object_id"]
GraphFilter(
    GraphFilterConfig(mode="nodes", node_match=[largest_id])
).run(G)


2026-07-06 15:27:55,364 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:27:55,409 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:27:55,417 - INFO - Finished in state Completed()


['81fe01bd949b495691201a5f0dd87ce8']

In [49]:
incident = GraphFilter(
    GraphFilterConfig(mode="edges", node_match=[largest_id])
).run(G)
pd.DataFrame(incident).head()


2026-07-06 15:27:57,585 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:27:57,629 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:27:57,636 - INFO - Finished in state Completed()


""


In [43]:
label_value = G.node_attrs(largest_id)["label"]
same_label_edges = GraphFilter(
    GraphFilterConfig(
        mode="edges",
        node_match={
            "source": {"label": label_value},
            "target": {"label": label_value},
        },
    )
).run(G)
len(same_label_edges)


KeyError: 'label'

In [ ]:
net = Network(notebook=True, cdn_resources='remote', bgcolor="#222222", font_color="white", select_menu=True)
net.barnes_hut()

# Convert the networkx object
net.from_nx(G.raw)
neighbor_map = net.get_adj_list()

# add neighbor data to node hover data
for node in net.nodes:
    node["title"] = node["id"] + "\n" +"  Neighbors:\n" + "\n".join(neighbor_map[node["id"]])
    node["value"] = len(neighbor_map[node["id"]])

# Render
net.show("nx_graph.html")

In [ ]:
net.show_buttons(filter_=['physics'])